# HE-IFD — start-once unattended fine-tuning sweep (template)

**One job. Three cells. Then sleep.**

This is the reusable template every headline / ablation experiment (ft04–ft10)
is produced from. The structure is fixed and never changes:

1. **Setup** (run once, no input) — clone/refresh the repo, install deps,
   prefetch the chosen backbone weights + HF datasets into the cache, set the
   offline env vars.
2. **Config** — the **only** interactive cell. An `ipywidgets` form (or a plain
   `input()` / dict block when headless) collects **every** choice and a single
   **"Confirm config"** button freezes them into one `CONFIG` dict.
3. **Run** (no input) — `run_unattended(CONFIG)` builds the full grid and runs it
   to completion: resumable (skips finished cells), error-tolerant (logs +
   continues, `failures.jsonl`), one heartbeat line per cell, periodic
   `results.csv` + README flush, and a final summary.

**The contract: there is ZERO interaction after you press "Confirm config".**
Start it and walk away — a VM restart resumes instead of restarting.

> Environment: a Colab GPU runtime (T4 is fine). Set **Runtime ▸ Change runtime
> type ▸ T4 GPU**. The repo is public (no token needed). Files land under
> `/content/HE-IFD`; download `results/<case>/` from the VS Code remote Explorer.

## Cell 1 · Setup — clone/refresh repo, install deps, prefetch weights & data

No input. Idempotent: safe to re-run after a VM reset (it refreshes code from
`origin/master` without clobbering `results/` or `cache/`). Edit
`PREFETCH_BACKBONES` / `PREFETCH_TEXT` / `PREFETCH_VISION` to match what your
Config will sweep — this is the ONE sanctioned download step; compute then runs
fully offline (`HF_HUB_OFFLINE=1`).

In [ ]:
# --- Cell 1: SETUP (no input) ---------------------------------------------
import os, sys, subprocess

REPO_DIR = "/content/HE-IFD"
REPO_URL = "https://github.com/hkanpak21/HE-IFD.git"

# 1. Clone once; on re-run just refresh CODE from origin (a full `git pull`
#    would abort because runs leave results/ dirty). Leaves results/ + cache/.
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=False)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "-q", "origin", "master"], check=False)
subprocess.run(["git", "checkout", "-q", "origin/master", "--",
                "src", "jobs", "tests"], check=False)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# 2. Deps (no-op if already present in the Colab image).
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "transformers", "datasets", "timm", "ipywidgets"], check=False)

# 3. PREFETCH — the ONE sanctioned download step. List what your Config sweeps so
#    the weights + datasets land in the cache; compute then runs offline.
PREFETCH_VISION = ["cifar100"]   # subset of: mnist, fashionmnist, cifar10, cifar100
PREFETCH_TEXT   = []             # subset of HF text datasets, e.g. ["ag_news", "dbpedia_14"]
PREFETCH_HF_MODELS = []          # e.g. ["roberta-base", "distilbert-base-uncased"]

import torch
print("CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

try:
    import torchvision as tv
    _vmap = {
        "mnist": tv.datasets.MNIST, "fashionmnist": tv.datasets.FashionMNIST,
        "cifar10": tv.datasets.CIFAR10, "cifar100": tv.datasets.CIFAR100,
    }
    for name in PREFETCH_VISION:
        ds = _vmap[name]
        ds("data", train=True, download=True); ds("data", train=False, download=True)
        print("prefetched vision:", name)
except Exception as e:
    print("vision prefetch note:", repr(e))

try:
    if PREFETCH_TEXT:
        from datasets import load_dataset
        for name in PREFETCH_TEXT:
            load_dataset(name); print("prefetched text dataset:", name)
    if PREFETCH_HF_MODELS:
        from transformers import AutoModel, AutoTokenizer
        for m in PREFETCH_HF_MODELS:
            AutoTokenizer.from_pretrained(m); AutoModel.from_pretrained(m)
            print("prefetched HF model:", m)
except Exception as e:
    print("HF prefetch note:", repr(e))

# 4. Offline env for the compute that follows (caches are now warm).
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("setup done — code at origin/master, caches warm, offline mode set.")

## Cell 2 · Config — **the only interactive cell**

Pick every choice here. Pressing **Confirm config** freezes them into a single
`CONFIG` dict — nothing downstream prompts you again. If `ipywidgets` is
unavailable (headless / non-Jupyter frontend) this cell falls back to a plain
`input()` prompt, and if even stdin is unavailable it uses the typed defaults so
an automated/AFK run still proceeds with **zero interaction**.

In [ ]:
# --- Cell 2: CONFIG (THE ONLY INTERACTIVE CELL) ---------------------------
# Pressing "Confirm config" freezes EVERY choice into the CONFIG dict below.
# Headless fallback: a single input() block, then the typed defaults.
from src.notebook_runner import DEFAULT_CONFIG, build_grid

# Defaults shown in the form (lists are comma-joined for the text inputs).
_D = dict(DEFAULT_CONFIG)
CONFIG = None  # set by Confirm (widgets) or the fallback below

def _csv(xs):
    return ",".join(str(x) for x in xs)

def _parse_list(s, cast):
    out = []
    for tok in str(s).split(","):
        tok = tok.strip()
        if not tok:
            continue
        out.append(cast(tok))
    return out

def _freeze(values):
    # Turn the raw form/string values into the typed CONFIG dict (the SINGLE
    # source of truth the Run cell consumes).
    cfg = dict(DEFAULT_CONFIG)
    cfg.update({
        "backbones": _parse_list(values["backbones"], str),
        "Ns":        _parse_list(values["Ns"], int),
        "alphas":    _parse_list(values["alphas"], float),
        "methods":   _parse_list(values["methods"], str),
        "seeds":     _parse_list(values["seeds"], int),
        "Ks":        _parse_list(values["Ks"], int),
        "local_steps":     _parse_list(values["local_steps"], str),
        "trainable_units": _parse_list(values["trainable_units"], str),
        "tau":        float(values["tau"]),
        "student_lr": float(values["student_lr"]),
        "agg_method": str(values["agg_method"]).strip(),
        "optimizer":  str(values["optimizer"]).strip(),
        "case":         str(values["case"]).strip(),
        "results_root": str(values["results_root"]).strip(),
        "resume":       bool(values["resume"]),
        "per_cell_timeout_sec": int(values["per_cell_timeout_sec"]),
        "flush_every":  int(values["flush_every"]),
    })
    ps = str(values.get("probe_size", "")).strip()
    cfg["probe_size"] = int(ps) if ps and ps.lower() != "none" else None
    return cfg

_defaults = {
    "backbones": _csv(_D["backbones"]), "Ns": _csv(_D["Ns"]),
    "alphas": _csv(_D["alphas"]), "methods": _csv(_D["methods"]),
    "seeds": _csv(_D["seeds"]), "Ks": _csv(_D["Ks"]),
    "local_steps": _csv(_D["local_steps"]), "trainable_units": _csv(_D["trainable_units"]),
    "tau": _D["tau"], "student_lr": _D["student_lr"],
    "agg_method": _D["agg_method"], "optimizer": _D["optimizer"],
    "probe_size": "" if _D["probe_size"] is None else _D["probe_size"],
    "case": _D["case"], "results_root": _D["results_root"],
    "resume": _D["resume"], "per_cell_timeout_sec": _D["per_cell_timeout_sec"],
    "flush_every": _D["flush_every"],
}

def _summarise(cfg):
    print("CONFIG frozen:")
    for k in ("case", "backbones", "Ns", "alphas", "methods", "seeds", "Ks",
              "local_steps", "trainable_units", "tau", "student_lr", "agg_method", "optimizer",
              "probe_size", "resume", "per_cell_timeout_sec", "flush_every"):
        print(f"  {k:22s} = {cfg[k]}")
    print(f"  GRID SIZE              = {len(build_grid(cfg))} cells")
    print("=> proceed to Cell 3 (Run). No further interaction.")

# --- Path A: ipywidgets interactive form ----------------------------------
try:
    import ipywidgets as W
    from IPython.display import display

    _w = {
        "backbones": W.Text(value=_defaults["backbones"], description="backbones"),
        "Ns": W.Text(value=_defaults["Ns"], description="Ns"),
        "alphas": W.Text(value=_defaults["alphas"], description="alphas"),
        "methods": W.Text(value=_defaults["methods"], description="methods"),
        "seeds": W.Text(value=_defaults["seeds"], description="seeds"),
        "Ks": W.Text(value=_defaults["Ks"], description="K list"),
        "local_steps": W.Text(value=_defaults["local_steps"], description="local_steps"),
        "trainable_units": W.Text(value=_defaults["trainable_units"], description="trainable_units"),
        "tau": W.FloatText(value=float(_defaults["tau"]), description="tau"),
        "student_lr": W.FloatText(value=float(_defaults["student_lr"]), description="student_lr"),
        "agg_method": W.Text(value=_defaults["agg_method"], description="agg_method"),
        "optimizer": W.Text(value=_defaults["optimizer"], description="optimizer"),
        "probe_size": W.Text(value=str(_defaults["probe_size"]), description="probe_size"),
        "case": W.Text(value=_defaults["case"], description="case"),
        "results_root": W.Text(value=_defaults["results_root"], description="results_root"),
        "resume": W.Checkbox(value=bool(_defaults["resume"]), description="resume"),
        "per_cell_timeout_sec": W.IntText(value=int(_defaults["per_cell_timeout_sec"]),
                                          description="cell_timeout_s"),
        "flush_every": W.IntText(value=int(_defaults["flush_every"]), description="flush_every"),
    }
    _btn = W.Button(description="Confirm config", button_style="success",
                    icon="check")
    _out = W.Output()

    def _on_confirm(_):
        global CONFIG
        with _out:
            _out.clear_output()
            CONFIG = _freeze({k: w.value for k, w in _w.items()})
            _summarise(CONFIG)
    _btn.on_click(_on_confirm)
    display(W.VBox(list(_w.values()) + [_btn, _out]))
    print("Fill the form, then press 'Confirm config'. CONFIG is set on click.")

# --- Path B: headless fallback (input(), then defaults) -------------------
except Exception as _e:
    print(f"[config] ipywidgets unavailable ({_e!r}); headless fallback.")
    raw = dict(_defaults)
    try:
        # One non-blocking-ish prompt; empty / EOF keeps the defaults. A truly
        # headless/AFK run (no stdin) drops straight through to the defaults so
        # the notebook still runs with ZERO interaction.
        _ans = input("Edit CONFIG dict in code? press Enter to accept defaults: ")
        if _ans.strip():
            print("Override the CONFIG values directly in this cell, then re-run.")
    except (EOFError, OSError):
        print("[config] no stdin — using typed defaults (AFK mode).")
    CONFIG = _freeze(raw)
    _summarise(CONFIG)

## Cell 3 · Run — `run_unattended(CONFIG)` (no input)

No input. Builds the grid from the frozen `CONFIG`, runs every cell, skips any
already-finished cell, logs + continues past per-cell failures (`failures.jsonl`),
prints one heartbeat line per cell, periodically flushes `results.csv` + README,
and prints a final summary. Re-running this cell after a VM reset **resumes**.

In [ ]:
# --- Cell 3: RUN (no input) -----------------------------------------------
assert CONFIG is not None, (
    "CONFIG is not set — run Cell 2 and press 'Confirm config' (or let the "
    "headless fallback set it) before running this cell."
)
from src.notebook_runner import run_unattended

summary = run_unattended(CONFIG)

# Show the freshly-written case README so partial/final results are visible here.
from pathlib import Path
_readme = Path(summary["readme"])
print("\n" + "=" * 70)
print(_readme.read_text() if _readme.exists() else "(no README written)")